## CELL 1: CONFIGURATION - Edit This Section

**Modify the CONFIG dictionary below with your simulation details.**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import io
import gc
from scipy import spatial
from collections import defaultdict

# ============================================================================
# USER CONFIGURATION: Modify values below for your protein analysis
# ============================================================================

CONFIG = {
    # Project Details
    "protein_name": "MyProtein",  # e.g., "KDEL", "GPCR", "Aquaporin"
    "project_name": "simulation_analysis",  # Identifier for output folder
    
    # Output Directory
    "output_dir": "./analysis_results",  # Where to save all outputs
    
    # Systems to Analyze
    "systems": {
        "system_1": {
            "label": "Apo State",  # Display name
            "topology": "./data/system1/structure.gro",  # .gro or .pdb
            "trajectory": "./data/system1/trajectory.xtc",  # .xtc or .dcd
            "color": "#FF6B6B",  # Hex color for plots
        },
        "system_2": {
            "label": "Bound State",
            "topology": "./data/system2/structure.gro",
            "trajectory": "./data/system2/trajectory.xtc",
            "color": "#4169E1",
        },
    },
    
    # ========== ANALYSIS CONFIGURATION ==========
    
    # 1. RMSD/RMSF Analysis
    "rmsd_rmsf": {
        "enabled": True,
        "atom_selection": "protein and backbone",  # MDAnalysis selection
        "reference_frame": 0,  # Use first frame as reference
    },
    
    # 2. Inter-helical Distance & Angle
    "inter_helical": {
        "enabled": True,
        "helix_pairs": [],  # Examples: [('H1', 'H2'), ('H2', 'H3')]
        # If empty, will use helices from secondary_structure below
    },
    
    # 3. Pore Hydration Analysis
    "pore_hydration": {
        "enabled": True,
        "pore_residues": [],  # Examples: [50, 51, 52, 100, 101]  OR leave empty to disable
        "water_selection": "water",
        "distance_cutoff": 5.0,  # Angstroms
    },
    
    # 4. SASA (Solvent Accessible Surface Area)
    "sasa": {
        "enabled": True,
        "frame_stride": 10,  # Process every 10th frame
    },
    
    # 5. Water Bridges (Direct & Mediated)
    "water_bridges": {
        "enabled": True,
        "residue_pairs": [],  # Examples: [(10, 50), (20, 100)]  OR leave empty to disable
        "hbond_distance_cutoff": 3.5,
        "occupancy_threshold": 0.01,  # 1% occupancy minimum
    },
    
    # 6. Minimum Distance Between Residues/Functional Groups
    "min_distance": {
        "enabled": True,
        "residue_pairs": [],  # Examples: [(1, 50), (20, 80)]  OR leave empty to disable
        "selection1": "protein and name CA",  # Selection 1 (default: CA atoms)
        "selection2": "protein and name CA",  # Selection 2
    },
    
    # 7. Hydrogen Bond Analysis
    "hbond": {
        "enabled": True,
        "distance_cutoff": 3.5,  # Angstroms
        "angle_cutoff": 120.0,  # Degrees
        "residue_pairs": [],  # Examples: [(10, 50)]  OR leave empty for all H-bonds
    },
    
    # Secondary Structure Definition (for annotations)
    "secondary_structure": {
        "helices": [],  # Examples: [(3, 26, 'H1'), (34, 52, 'H2')]
        "strands": [],  # Examples: [(10, 20, 'S1')]
    },
    
    # Visualization Settings
    "plotting": {
        "rolling_average_ns": 10.0,
        "last_frames_ns": 500,
        "dpi": 300,
        "figure_style": "seaborn-v0_8-whitegrid",
    },
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"✅ Configuration loaded.")
print(f"   Protein: {CONFIG['protein_name']}")
print(f"   Output: {CONFIG['output_dir']}")
print(f"   Systems: {list(CONFIG['systems'].keys())}")

## CELL 2: IMPORT LIBRARIES & UTILITY FUNCTIONS

In [ ]:
# Import analysis libraries
try:
    import MDAnalysis as mda
    from MDAnalysis.analysis import distances, hydration
    print("✅ MDAnalysis imported")
except ImportError as e:
    print(f"❌ MDAnalysis not found: {e}")
    print("   Install with: conda install -c conda-forge mdanalysis")

try:
    import mdtraj as md
    print("✅ MDTraj imported")
except ImportError as e:
    print(f"❌ MDTraj not found: {e}")
    print("   Install with: conda install -c conda-forge mdtraj")

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def get_color_palette(system_dict):
    """Extract color mapping from system definitions."""
    return {k: v.get("color", "#000000") for k, v in system_dict.items()}

def get_system_labels(system_dict):
    """Extract human-readable labels from system definitions."""
    return {k: v.get("label", k) for k, v in system_dict.items()}

def validate_files(config):
    """Validate that all trajectory files exist."""
    missing = []
    for sys_key, sys_info in config["systems"].items():
        if not os.path.exists(sys_info["topology"]):
            missing.append(f"  ❌ {sys_key} topology: {sys_info['topology']}")
        if not os.path.exists(sys_info["trajectory"]):
            missing.append(f"  ❌ {sys_key} trajectory: {sys_info['trajectory']}")
    
    if missing:
        print("⚠️ Some files not found:")
        for msg in missing:
            print(msg)
        return False
    else:
        print("✅ All trajectory files verified")
        return True

def read_xvg(filepath, column_names):
    """Read GROMACS .xvg files."""
    clean_lines = []
    with open(filepath, 'r') as f:
        for line in f:
            if not line.strip().startswith(('@', '#')):
                clean_lines.append(line)
    string_io = io.StringIO("".join(clean_lines))
    return pd.read_csv(string_io, header=None, delim_whitespace=True, names=column_names)

print("✅ Utility functions loaded")
validate_files(CONFIG)

## CELL 3: RMSD & RMSF ANALYSIS

Measures overall protein stability (RMSD) and per-residue flexibility (RMSF).

In [ ]:
def calculate_rmsd_rmsf(config):
    """Calculate RMSD and RMSF for all systems."""
    if not config["rmsd_rmsf"]["enabled"]:
        print("⊘ RMSD/RMSF disabled")
        return None, None
    
    print("\n" + "="*60)
    print("1. RMSD & RMSF ANALYSIS")
    print("="*60)
    
    atom_sel = config["rmsd_rmsf"]["atom_selection"]
    ref_frame = config["rmsd_rmsf"]["reference_frame"]
    out_dir = f"{config['output_dir']}/rmsd_rmsf"
    os.makedirs(out_dir, exist_ok=True)
    
    all_rmsd = []
    all_rmsf = []
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            atoms = u.select_atoms(atom_sel)
            
            if len(atoms) == 0:
                print(f"  ⚠️ No atoms selected")
                continue
            
            # Reference coordinates
            u.trajectory[ref_frame]
            ref_coords = atoms.positions.copy()
            
            # RMSD calculation
            rmsd_vals = []
            times = []
            deviations = []
            
            for frame in u.trajectory:
                rmsd = np.sqrt(np.mean((atoms.positions - ref_coords)**2))
                rmsd_vals.append(rmsd)
                times.append(u.trajectory.time / 1000.0)  # ps -> ns
                deviations.append(atoms.positions - ref_coords)
            
            # Store RMSD
            df_rmsd = pd.DataFrame({
                'Time_ns': times,
                'RMSD_A': rmsd_vals,
                'System': sys_key
            })
            all_rmsd.append(df_rmsd)
            
            # RMSF calculation (per-residue)
            deviations = np.array(deviations)
            rmsf_vals = np.sqrt(np.mean(deviations**2, axis=0))
            
            residue_nums = []
            per_res_rmsf = []
            
            for residue in atoms.residues:
                res_atom_idx = [a.index for a in residue.atoms if a in atoms]
                if res_atom_idx:
                    res_rmsf = np.mean(rmsf_vals[res_atom_idx])
                    residue_nums.append(residue.resnum)
                    per_res_rmsf.append(res_rmsf)
            
            df_rmsf = pd.DataFrame({
                'Residue': residue_nums,
                'RMSF_A': per_res_rmsf,
                'System': sys_key
            })
            all_rmsf.append(df_rmsf)
            
            print(f"  ✅ RMSD: {min(rmsd_vals):.2f} - {max(rmsd_vals):.2f} Å")
            print(f"  ✅ RMSF: {min(per_res_rmsf):.2f} - {max(per_res_rmsf):.2f} Å")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    # Save data
    if all_rmsd:
        pd.concat(all_rmsd).to_csv(f"{out_dir}/rmsd_all.csv", index=False)
    if all_rmsf:
        pd.concat(all_rmsf).to_csv(f"{out_dir}/rmsf_all.csv", index=False)
    
    return all_rmsd, all_rmsf

# Run analysis
rmsd_data, rmsf_data = calculate_rmsd_rmsf(CONFIG)

## CELL 4: PLOT RMSD & RMSF

In [ ]:
def plot_rmsd(rmsd_data, config):
    """Plot comparative RMSD."""
    if not rmsd_data:
        return
    
    df = pd.concat(rmsd_data, ignore_index=True)
    colors = get_color_palette(config["systems"])
    labels = get_system_labels(config["systems"])
    
    print("\nPlotting RMSD...")
    
    sns.set_style(config["plotting"]["figure_style"])
    fig, ax = plt.subplots(figsize=(14, 6))
    
    window_ns = config["plotting"]["rolling_average_ns"]
    
    for sys_key in config["systems"].keys():
        df_sys = df[df['System'] == sys_key].copy()
        if df_sys.empty:
            continue
        
        dt = (df_sys['Time_ns'].iloc[1] - df_sys['Time_ns'].iloc[0]) if len(df_sys) > 1 else 1.0
        window_frames = max(1, int(window_ns / dt))
        df_sys['smooth'] = df_sys['RMSD_A'].rolling(window_frames, center=True, min_periods=1).mean()
        
        last_ns = config["plotting"]["last_frames_ns"]
        df_recent = df_sys[df_sys['Time_ns'] >= df_sys['Time_ns'].max() - last_ns]
        mean_val = df_recent['RMSD_A'].mean()
        std_val = df_recent['RMSD_A'].std()
        
        label = f"{labels[sys_key]} (avg: {mean_val:.2f} ± {std_val:.2f} Å)"
        color = colors[sys_key]
        
        ax.plot(df_sys['Time_ns'], df_sys['RMSD_A'], color=color, alpha=0.2, linewidth=0.5)
        ax.plot(df_sys['Time_ns'], df_sys['smooth'], color=color, linewidth=2.5, label=label)
    
    ax.set_xlabel("Time (ns)", fontsize=12, fontweight='bold')
    ax.set_ylabel("RMSD (Å)", fontsize=12, fontweight='bold')
    ax.set_title(f"RMSD Analysis: {config['protein_name']}", fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    outpath = f"{config['output_dir']}/rmsd_rmsf/rmsd_plot.png"
    plt.savefig(outpath, dpi=config["plotting"]["dpi"])
    print(f"✅ Saved: {outpath}")
    plt.show()

def plot_rmsf(rmsf_data, config):
    """Plot comparative RMSF."""
    if not rmsf_data:
        return
    
    df = pd.concat(rmsf_data, ignore_index=True)
    colors = get_color_palette(config["systems"])
    labels = get_system_labels(config["systems"])
    
    print("\nPlotting RMSF...")
    
    sns.set_style(config["plotting"]["figure_style"])
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for sys_key in config["systems"].keys():
        df_sys = df[df['System'] == sys_key]
        if df_sys.empty:
            continue
        ax.plot(df_sys['Residue'], df_sys['RMSF_A'], color=colors[sys_key], linewidth=2.5, label=labels[sys_key])
    
    # Add secondary structure
    if config["secondary_structure"]["helices"]:
        helix_colors = ['gold', 'lightcoral', 'plum', 'lightskyblue', 'mediumpurple']
        max_rmsf = df['RMSF_A'].max()
        
        for i, (start, end, name) in enumerate(config["secondary_structure"]["helices"]):
            color = helix_colors[i % len(helix_colors)]
            ax.axvspan(start, end, color=color, alpha=0.2, zorder=0)
            ax.text((start + end) / 2, max_rmsf * 0.95, name, ha='center', va='top', fontsize=9)
    
    ax.set_xlabel("Residue Number", fontsize=12, fontweight='bold')
    ax.set_ylabel("RMSF (Å)", fontsize=12, fontweight='bold')
    ax.set_title(f"RMSF Analysis: {config['protein_name']}", fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    outpath = f"{config['output_dir']}/rmsd_rmsf/rmsf_plot.png"
    plt.savefig(outpath, dpi=config["plotting"]["dpi"])
    print(f"✅ Saved: {outpath}")
    plt.show()

# Generate plots
if rmsd_data:
    plot_rmsd(rmsd_data, CONFIG)
if rmsf_data:
    plot_rmsf(rmsf_data, CONFIG)

## CELL 5: INTER-HELICAL DISTANCE & ANGLE

Analyzes the distance and angle between helical domains.

In [ ]:
def calculate_inter_helical_dynamics(config):
    """Calculate inter-helical distance and angle dynamics."""
    if not config["inter_helical"]["enabled"]:
        print("⊘ Inter-helical analysis disabled")
        return None
    
    print("\n" + "="*60)
    print("2. INTER-HELICAL DISTANCE & ANGLE")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/inter_helical"
    os.makedirs(out_dir, exist_ok=True)
    
    # Get helix pairs from config or secondary structure
    helix_pairs = config["inter_helical"]["helix_pairs"]
    if not helix_pairs and config["secondary_structure"]["helices"]:
        helices = config["secondary_structure"]["helices"]
        helix_pairs = [(helices[i][2], helices[j][2]) 
                      for i in range(len(helices)) 
                      for j in range(i+1, len(helices))]
    
    if not helix_pairs:
        print("⚠️ No helix pairs defined. Skipping inter-helical analysis.")
        return None
    
    results = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            
            # Build helix residue ranges from config
            helix_residues = {}
            for start, end, name in config["secondary_structure"]["helices"]:
                helix_residues[name] = (start, end)
            
            # Calculate distances and angles
            times = []
            distances_dict = defaultdict(list)
            angles_dict = defaultdict(list)
            
            for frame in u.trajectory:
                times.append(u.trajectory.time / 1000.0)  # ps -> ns
                
                for h1, h2 in helix_pairs:
                    if h1 not in helix_residues or h2 not in helix_residues:
                        continue
                    
                    start1, end1 = helix_residues[h1]
                    start2, end2 = helix_residues[h2]
                    
                    # Get CA atoms of each helix
                    sel1 = u.select_atoms(f"resid {start1}:{end1} and name CA")
                    sel2 = u.select_atoms(f"resid {start2}:{end2} and name CA")
                    
                    if len(sel1) > 0 and len(sel2) > 0:
                        # Center of mass
                        com1 = sel1.center_of_mass()
                        com2 = sel2.center_of_mass()
                        distance = np.linalg.norm(com2 - com1)
                        distances_dict[f"{h1}-{h2}"].append(distance)
                        
                        # Vector angle
                        vec = com2 - com1
                        angle = np.arctan2(vec[1], vec[0]) * 180.0 / np.pi
                        angles_dict[f"{h1}-{h2}"].append(angle)
            
            # Store results
            results[sys_key] = {
                'times': times,
                'distances': distances_dict,
                'angles': angles_dict
            }
            print(f"  ✅ Calculated {len(distances_dict)} helix pair distances")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    # Plot results
    if results:
        plot_inter_helical(results, config)
    
    return results

def plot_inter_helical(results, config):
    """Plot inter-helical distance and angle dynamics."""
    print("\nPlotting inter-helical dynamics...")
    
    colors = get_color_palette(config["systems"])
    labels = get_system_labels(config["systems"])
    
    # Get all helix pairs
    all_pairs = set()
    for sys_result in results.values():
        all_pairs.update(sys_result['distances'].keys())
    
    for pair_name in all_pairs:
        # Distance plot
        fig, ax = plt.subplots(figsize=(12, 5))
        
        for sys_key, result in results.items():
            if pair_name in result['distances']:
                ax.plot(result['times'], result['distances'][pair_name], 
                       label=labels[sys_key], color=colors[sys_key], linewidth=2)
        
        ax.set_xlabel("Time (ns)", fontsize=11, fontweight='bold')
        ax.set_ylabel("Distance (Å)", fontsize=11, fontweight='bold')
        ax.set_title(f"Inter-helical Distance: {pair_name}", fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        outpath = f"{config['output_dir']}/inter_helical/{pair_name}_distance.png"
        plt.savefig(outpath, dpi=config["plotting"]["dpi"])
        plt.show()
    
    print(f"✅ Plots saved")

# Run analysis
inter_helical_results = calculate_inter_helical_dynamics(CONFIG)

## CELL 6: SASA ANALYSIS (Solvent Accessible Surface Area)

In [ ]:
def calculate_sasa(config):
    """Calculate SASA for all systems using MDTraj."""
    if not config["sasa"]["enabled"]:
        print("⊘ SASA analysis disabled")
        return None
    
    print("\n" + "="*60)
    print("4. SASA ANALYSIS (Solvent Accessible Surface Area)")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/sasa"
    os.makedirs(out_dir, exist_ok=True)
    
    stride = config["sasa"]["frame_stride"]
    all_sasa_data = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing SASA: {sys_key}")
        
        try:
            traj = md.load(sys_info["trajectory"], top=sys_info["topology"], stride=stride)
            print(f"  Loaded {len(traj)} frames")
            
            # Calculate SASA using Shrake-Rupley
            sasa = md.shrake_rupley(traj, mode='residue')
            sasa_angstrom = sasa * 100.0  # nm^2 -> Angstrom^2
            
            # Create DataFrame
            df_dict = {'Time_ns': traj.time / 1000.0}  # ps -> ns
            
            for i, residue in enumerate(traj.topology.residues):
                res_name = f"{residue.name}_{residue.resSeq}"
                df_dict[res_name] = sasa_angstrom[:, i]
            
            df_sasa = pd.DataFrame(df_dict)
            df_sasa['System'] = sys_key
            all_sasa_data[sys_key] = df_sasa
            
            # Save
            csv_path = f"{out_dir}/{sys_key}_sasa.csv"
            df_sasa.to_csv(csv_path, index=False)
            print(f"  ✅ Saved: {csv_path}")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    return all_sasa_data

# Run analysis
sasa_data = calculate_sasa(CONFIG)

## CELL 7: PORE HYDRATION ANALYSIS

In [ ]:
def calculate_pore_hydration(config):
    """Calculate water occupancy in pore region."""
    if not config["pore_hydration"]["enabled"] or not config["pore_hydration"]["pore_residues"]:
        print("⊘ Pore hydration analysis disabled or no residues defined")
        return None
    
    print("\n" + "="*60)
    print("3. PORE HYDRATION ANALYSIS")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/pore_hydration"
    os.makedirs(out_dir, exist_ok=True)
    
    pore_residues = config["pore_hydration"]["pore_residues"]
    distance_cutoff = config["pore_hydration"]["distance_cutoff"]
    water_sel = config["pore_hydration"]["water_selection"]
    
    results = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            
            # Select pore and water
            pore_sel = u.select_atoms(f"resid {' '.join(map(str, pore_residues))}")
            water_atoms = u.select_atoms(water_sel)
            
            if len(pore_sel) == 0:
                print(f"  ⚠️ No pore residues found")
                continue
            
            times = []
            water_count = []
            water_ids = set()
            
            for frame in u.trajectory:
                times.append(u.trajectory.time / 1000.0)  # ps -> ns
                
                # Find waters within cutoff
                distances = spatial.distance.cdist(
                    water_atoms.positions,
                    pore_sel.positions
                )
                nearby_waters = np.where(np.min(distances, axis=1) < distance_cutoff)[0]
                water_count.append(len(nearby_waters))
                water_ids.update(water_atoms[nearby_waters].resids)
            
            df_hydration = pd.DataFrame({
                'Time_ns': times,
                'Water_Count': water_count,
                'System': sys_key
            })
            
            results[sys_key] = {
                'data': df_hydration,
                'avg_waters': np.mean(water_count),
                'water_ids': water_ids
            }
            
            df_hydration.to_csv(f"{out_dir}/{sys_key}_pore_hydration.csv", index=False)
            print(f"  ✅ Avg. waters in pore: {np.mean(water_count):.1f} ± {np.std(water_count):.1f}")
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    # Plot
    if results:
        plot_pore_hydration(results, CONFIG)
    
    return results

def plot_pore_hydration(results, config):
    """Plot pore hydration time series."""
    print("\nPlotting pore hydration...")
    
    colors = get_color_palette(config["systems"])
    labels = get_system_labels(config["systems"])
    
    fig, ax = plt.subplots(figsize=(12, 5))
    
    for sys_key, result in results.items():
        df = result['data']
        ax.plot(df['Time_ns'], df['Water_Count'], label=labels[sys_key], 
               color=colors[sys_key], linewidth=2, alpha=0.7)
    
    ax.set_xlabel("Time (ns)", fontsize=11, fontweight='bold')
    ax.set_ylabel("Number of Waters", fontsize=11, fontweight='bold')
    ax.set_title(f"Pore Hydration: {config['protein_name']}", fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    
    plt.tight_layout()
    outpath = f"{config['output_dir']}/pore_hydration/pore_hydration.png"
    plt.savefig(outpath, dpi=config["plotting"]["dpi"])
    plt.show()
    print(f"✅ Saved: {outpath}")

# Run analysis
pore_hydration_results = calculate_pore_hydration(CONFIG)

## CELL 8: WATER BRIDGES ANALYSIS (Direct & Mediated)

In [ ]:
def calculate_water_bridges(config):
    """Calculate direct and water-mediated hydrogen bridges."""
    if not config["water_bridges"]["enabled"] or not config["water_bridges"]["residue_pairs"]:
        print("⊘ Water bridges analysis disabled or no residue pairs defined")
        return None
    
    print("\n" + "="*60)
    print("5. WATER BRIDGES ANALYSIS (Direct & Mediated)")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/water_bridges"
    os.makedirs(out_dir, exist_ok=True)
    
    pairs = config["water_bridges"]["residue_pairs"]
    hbond_cutoff = config["water_bridges"]["hbond_distance_cutoff"]
    occ_threshold = config["water_bridges"]["occupancy_threshold"]
    
    results = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            water = u.select_atoms("water")
            
            bridge_summary = {}
            
            for res1_id, res2_id in pairs:
                res1 = u.select_atoms(f"resid {res1_id}")
                res2 = u.select_atoms(f"resid {res2_id}")
                
                if len(res1) == 0 or len(res2) == 0:
                    continue
                
                pair_key = f"{res1_id}-{res2_id}"
                direct_bridge = 0
                mediated_bridge = 0
                total_frames = 0
                
                for frame in u.trajectory:
                    total_frames += 1
                    
                    # Direct distance
                    dist = np.min(spatial.distance.cdist(res1.positions, res2.positions))
                    if dist < hbond_cutoff:
                        direct_bridge += 1
                    
                    # Water-mediated bridge
                    for water_mol in water.residues:
                        w_atoms = water_mol.atoms
                        d1 = np.min(spatial.distance.cdist(res1.positions, w_atoms.positions))
                        d2 = np.min(spatial.distance.cdist(res2.positions, w_atoms.positions))
                        if d1 < hbond_cutoff and d2 < hbond_cutoff:
                            mediated_bridge += 1
                            break
                
                direct_occ = direct_bridge / total_frames
                mediated_occ = mediated_bridge / total_frames
                
                if direct_occ >= occ_threshold or mediated_occ >= occ_threshold:
                    bridge_summary[pair_key] = {
                        'direct_occupancy': direct_occ,
                        'mediated_occupancy': mediated_occ,
                        'total_occupancy': direct_occ + mediated_occ
                    }
            
            results[sys_key] = bridge_summary
            print(f"  ✅ Found {len(bridge_summary)} active bridges")
            
            # Save summary
            df_bridges = pd.DataFrame(bridge_summary).T
            df_bridges.to_csv(f"{out_dir}/{sys_key}_water_bridges_summary.csv")
            

    # Print summary
    print("\n" + "-"*60)
    print("WATER BRIDGES SUMMARY")
    print("-"*60)
    
    for sys_key, bridges in results.items():
        print(f"\n{sys_key}:")
        if bridges:
            for pair, occ in bridges.items():
                print(f"  {pair}:")
                print(f"    Direct:   {occ['direct_occupancy']:.1%}")
                print(f"    Mediated: {occ['mediated_occupancy']:.1%}")
        else:
            print("  (No significant bridges)")
    
    return results

# Run analysis
water_bridges_results = calculate_water_bridges(CONFIG)

## CELL 9: MINIMUM DISTANCE ANALYSIS

In [ ]:
def calculate_minimum_distances(config):
    """Calculate minimum distances between residue pairs."""
    if not config["min_distance"]["enabled"] or not config["min_distance"]["residue_pairs"]:
        print("⊘ Minimum distance analysis disabled or no pairs defined")
        return None
    
    print("\n" + "="*60)
    print("6. MINIMUM DISTANCE ANALYSIS")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/min_distance"
    os.makedirs(out_dir, exist_ok=True)
    
    pairs = config["min_distance"]["residue_pairs"]
    sel1_str = config["min_distance"]["selection1"]
    sel2_str = config["min_distance"]["selection2"]
    
    results = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            
            times = []
            distances_dict = defaultdict(list)
            
            for frame in u.trajectory:
                times.append(u.trajectory.time / 1000.0)  # ps -> ns
                
                for res1_id, res2_id in pairs:
                    sel1 = u.select_atoms(f"resid {res1_id} and {sel1_str.split('and')[-1].strip()}")
                    sel2 = u.select_atoms(f"resid {res2_id} and {sel2_str.split('and')[-1].strip()}")
                    
                    if len(sel1) > 0 and len(sel2) > 0:
                        dist = np.min(spatial.distance.cdist(sel1.positions, sel2.positions))
                        distances_dict[f"{res1_id}-{res2_id}"].append(dist)
            
            results[sys_key] = {
                'times': times,
                'distances': distances_dict
            }
            
            print(f"  ✅ Calculated distances for {len(distances_dict)} residue pairs")
            
    
    # Plot
    if results:
        plot_minimum_distances(results, config)
    
    return results

def plot_minimum_distances(results, config):
    """Plot minimum distance dynamics."""
    print("\nPlotting minimum distances...")
    
    colors = get_color_palette(config["systems"])
    labels = get_system_labels(config["systems"])
    
    # Get all pairs
    all_pairs = set()
    for sys_result in results.values():
        all_pairs.update(sys_result['distances'].keys())
    
    for pair_name in all_pairs:
        fig, ax = plt.subplots(figsize=(12, 5))
        
        for sys_key, result in results.items():
            if pair_name in result['distances']:
                ax.plot(result['times'], result['distances'][pair_name], 
                       label=labels[sys_key], color=colors[sys_key], linewidth=1.5, alpha=0.8)
        
        ax.set_xlabel("Time (ns)", fontsize=11, fontweight='bold')
        ax.set_ylabel("Distance (Å)", fontsize=11, fontweight='bold')
        ax.set_title(f"Minimum Distance: Residues {pair_name}", fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)
        
        plt.tight_layout()
        outpath = f"{config['output_dir']}/min_distance/{pair_name}_distance.png"
        plt.savefig(outpath, dpi=config["plotting"]["dpi"])
        plt.show()
    
    print(f"✅ Distance plots saved")

# Run analysis
min_distance_results = calculate_minimum_distances(CONFIG)

## CELL 10: HYDROGEN BOND ANALYSIS

In [ ]:
def calculate_hydrogen_bonds(config):
    """Calculate persistent hydrogen bonds."""
    if not config["hbond"]["enabled"]:
        print("⊘ Hydrogen bond analysis disabled")
        return None
    
    print("\n" + "="*60)
    print("7. HYDROGEN BOND ANALYSIS")
    print("="*60)
    
    out_dir = f"{config['output_dir']}/hydrogen_bonds"
    os.makedirs(out_dir, exist_ok=True)
    
    distance_cutoff = config["hbond"]["distance_cutoff"]
    angle_cutoff = config["hbond"]["angle_cutoff"]
    specific_pairs = config["hbond"]["residue_pairs"]
    
    results = {}
    
    for sys_key, sys_info in config["systems"].items():
        print(f"\nProcessing: {sys_key}")
        
        try:
            u = mda.Universe(sys_info["topology"], sys_info["trajectory"])
            
            # Define H-bond donors and acceptors
            donors = u.select_atoms("((name N or name O) and (protein or residue HOH))")
            acceptors = u.select_atoms("((name O or name N) and (protein or residue HOH))")
            
            hbond_counts = defaultdict(int)
            hbond_occupancy = defaultdict(int)
            total_frames = 0
            
            for frame in u.trajectory:
                total_frames += 1
                
                # Calculate donor-acceptor distances
                distances = spatial.distance.cdist(donors.positions, acceptors.positions)
                
                # Find H-bonds
                h_bonds = np.where(distances < distance_cutoff)
                
                for donor_idx, acc_idx in zip(h_bonds[0], h_bonds[1]):
                    donor_res = donors[donor_idx].resnum
                    acc_res = acceptors[acc_idx].resnum
                    
                    # Filter by specific pairs if provided
                    if specific_pairs and (donor_res, acc_res) not in specific_pairs:
                        continue
                    
                    pair_key = f"{donor_res}-{acc_res}"
                    hbond_occupancy[pair_key] += 1
                    hbond_counts[pair_key] += 1
            
            # Calculate occupancy
            for pair_key in hbond_occupancy:
                hbond_occupancy[pair_key] = hbond_occupancy[pair_key] / total_frames
            
            results[sys_key] = {
                'occupancy': hbond_occupancy,
                'total_frames': total_frames
            }
            
            # Save results
            df_hbonds = pd.DataFrame([
                {'Residue_Pair': pair, 'Occupancy': occ}
                for pair, occ in hbond_occupancy.items()
            ])
            df_hbonds = df_hbonds.sort_values('Occupancy', ascending=False)
            df_hbonds.to_csv(f"{out_dir}/{sys_key}_hbonds.csv", index=False)
            
            print(f"  ✅ Found {len(hbond_occupancy)} H-bonds")
            print(f"     Top 5 H-bonds:")
            for idx, row in df_hbonds.head(5).iterrows():
                print(f"       {row['Residue_Pair']}: {row['Occupancy']:.1%}")
            
    
    return results

# Run analysis
hbond_results = calculate_hydrogen_bonds(CONFIG)

## CELL 11: GENERATE SUMMARY REPORT

In [ ]:
def generate_summary_report(config, results_dict):
    """Generate HTML summary report of all analyses."""
    print("\n" + "="*60)
    print("GENERATING SUMMARY REPORT")
    print("="*60)
    
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Analysis Report: {config['protein_name']}</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 40px; }}
            h1 {{ color: #333; border-bottom: 2px solid #007bff; }}
            h2 {{ color: #555; margin-top: 30px; }}
            .summary {{ background-color: #f8f9fa; padding: 15px; border-radius: 5px; }}
            table {{ border-collapse: collapse; width: 100%; margin-top: 10px; }}
            th, td {{ border: 1px solid #ddd; padding: 12px; text-align: left; }}
            th {{ background-color: #007bff; color: white; }}
        </style>
    </head>
    <body>
        <h1>Comprehensive MD Analysis Report</h1>
        <p><strong>Protein:</strong> {config['protein_name']}</p>
        <p><strong>Analysis Date:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        
        <h2>Systems Analyzed</h2>
        <div class="summary">
    """
    
    for sys_key, sys_info in config["systems"].items():
        html_content += f"""
            <p><strong>{sys_info['label']}:</strong> {sys_info['trajectory']}</p>
        """
    
    html_content += """
        </div>
        
        <h2>Analyses Performed</h2>
        <ul>
    """
    
    analyses = [
        ("rmsd_rmsf", "RMSD & RMSF"),
        ("inter_helical", "Inter-helical Dynamics"),
        ("pore_hydration", "Pore Hydration"),
        ("sasa", "SASA Analysis"),
        ("water_bridges", "Water Bridges"),
        ("min_distance", "Minimum Distances"),
        ("hbond", "Hydrogen Bonds"),
    ]
    
    for config_key, analysis_name in analyses:
        if config[config_key]["enabled"]:
            html_content += f"<li>✅ {analysis_name}</li>"
        else:
            html_content += f"<li>⊘ {analysis_name}</li>"
    
    html_content += """
        </ul>
        
        <h2>Output Files</h2>
        <p>All results saved to: <code>" + config['output_dir'] + """</code></p>
        <div class="summary">
            <ul>
                <li>rmsd_rmsf/ - RMSD & RMSF data and plots</li>
                <li>sasa/ - SASA per-residue data</li>
                <li>inter_helical/ - Inter-helical distance and angle dynamics</li>
                <li>pore_hydration/ - Water occupancy in pore region</li>
                <li>water_bridges/ - Bridge occupancy summary</li>
                <li>min_distance/ - Minimum distance time series</li>
                <li>hydrogen_bonds/ - H-bond occupancy tables</li>
            </ul>
        </div>
        
        <h2>Notes</h2>
        <p>For detailed analysis methodology, see the documentation in docs/ANALYSIS_GUIDE.md</p>
        
    </body>
    </html>
    """
    
    # Save report
    report_path = f"{config['output_dir']}/analysis_report.html"
    with open(report_path, 'w') as f:
        f.write(html_content)
    
    print(f"\n✅ Report generated: {report_path}")
    print("\n" + "="*60)
    print("ANALYSIS COMPLETE")
    print("="*60)

# Generate report
results_dict = {
    'rmsd_rmsf': (rmsd_data, rmsf_data),
    'inter_helical': inter_helical_results,
    'sasa': sasa_data,
    'pore_hydration': pore_hydration_results,
    'water_bridges': water_bridges_results,
    'min_distance': min_distance_results,
    'hbond': hbond_results,
}

generate_summary_report(CONFIG, results_dict)